# 1915(c) Waiver Extraction: HTML + Text Combined
Runs both extractors (72 columns each) and displays results.

**New variables added (Request Info 1 of 3):** title, approval_period, replacedwaiver, waiver_type, effective_date

Note: `approval_period` is a radio button with no indicator in text files, returns empty for text extraction.

In [ ]:
# ── Configuration ──
INPUT_DIR = "/Users/thomt/Documents/JHU/Bloomberg Public Health/source_codes/datasets/1915(c) waivers/"
OUTPUT_DIR = "./output"

# Single file tests
TEST_HTML = "/Users/thomt/Documents/JHU/Bloomberg Public Health/source_codes/datasets/1915(c) waivers/AK/AK.0260/AK0260R0600.htm"
TEST_TXT  = "/Users/thomt/Documents/JHU/Bloomberg Public Health/source_codes/datasets/1915(c) waivers/MO/MO.0026/MO0026R0900.txt"

import os, csv, re
from pathlib import Path
import pandas as pd
import numpy as np

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
from html_waiver_extractor_combined_v3 import (
    HTMLWaiverExtractor, ALL_COLUMNS as HTML_COLUMNS,
    load_html_document, extract_document_id
)
from text_waiver_extractor_combined_v2 import (
    TextWaiverExtractor, ALL_COLUMNS as TEXT_COLUMNS
)

print(f"HTML columns: {len(HTML_COLUMNS)}")
print(f"Text columns: {len(TEXT_COLUMNS)}")
print(f"Columns match: {HTML_COLUMNS == TEXT_COLUMNS}")

## 1. Single File Test: HTML

In [ ]:
doc = load_html_document(TEST_HTML)
html_ext = HTMLWaiverExtractor(extract_document_id(TEST_HTML), doc)
html_result = html_ext.extract_all()

print(f"Document: {html_result['document_id']}")
print(f"\n--- Request Info (1 of 3) ---")
for k in ['title', 'approval_period', 'replacedwaiver', 'waiver_type', 'effective_date']:
    v = html_result[k]
    print(f"  {k:25s} {repr(v)}")

filled = sum(1 for k, v in html_result.items() if k != 'document_id' and v != '' and v is not None)
print(f"\nTotal filled: {filled}/{len(html_result)-1}")

## 2. Single File Test: Text

In [ ]:
with open(TEST_TXT, 'r', encoding='utf-8', errors='replace') as f:
    text = f.read()
text = text.replace('\r\r', '\n').replace('\r', '\n')
lines = text.split('\n')

txt_ext = TextWaiverExtractor(Path(TEST_TXT).stem, lines)
txt_result = txt_ext.extract_all()

print(f"Document: {txt_result['document_id']}")
print(f"\n--- Request Info (1 of 3) ---")
for k in ['title', 'approval_period', 'replacedwaiver', 'waiver_type', 'effective_date']:
    v = txt_result[k]
    print(f"  {k:25s} {repr(v)}")

filled = sum(1 for k, v in txt_result.items() if k != 'document_id' and v != '' and v is not None)
print(f"\nTotal filled: {filled}/{len(txt_result)-1}")

## 3. Batch Extraction: HTML

In [ ]:
htm_files = sorted(
    list(Path(INPUT_DIR).rglob('*.htm')) + list(Path(INPUT_DIR).rglob('*.html'))
)
print(f"Found {len(htm_files)} HTML files")

html_results = []
html_errors = []

for i, fp in enumerate(htm_files):
    if (i + 1) % 100 == 0:
        print(f"  [{i+1}/{len(htm_files)}] Success: {len(html_results)}, Failed: {len(html_errors)}")
    try:
        doc_id = Path(fp).stem
        document = load_html_document(str(fp))
        ext = HTMLWaiverExtractor(doc_id, document)
        html_results.append(ext.extract_all())
    except Exception as e:
        html_errors.append({'file': str(fp), 'error': str(e)})

df_html = pd.DataFrame(html_results, columns=HTML_COLUMNS)
print(f"\nDone: {len(html_results)} success, {len(html_errors)} failed")
print(f"Shape: {df_html.shape}")

## 4. Batch Extraction: Text

In [ ]:
txt_files = sorted(Path(INPUT_DIR).rglob('*.txt'))
print(f"Found {len(txt_files)} text files")

txt_results = []
txt_errors = []

for i, fp in enumerate(txt_files):
    if (i + 1) % 100 == 0:
        print(f"  [{i+1}/{len(txt_files)}] Success: {len(txt_results)}, Failed: {len(txt_errors)}")
    try:
        doc_id = Path(fp).stem
        with open(fp, 'r', encoding='utf-8', errors='replace') as f:
            raw = f.read()
        raw = raw.replace('\r\r', '\n').replace('\r', '\n')
        lines = raw.split('\n')
        ext = TextWaiverExtractor(doc_id, lines)
        txt_results.append(ext.extract_all())
    except Exception as e:
        txt_errors.append({'file': str(fp), 'error': str(e)})

df_txt = pd.DataFrame(txt_results, columns=TEXT_COLUMNS)
print(f"\nDone: {len(txt_results)} success, {len(txt_errors)} failed")
print(f"Shape: {df_txt.shape}")

## 5. Fill-Rate Summary (Side by Side)

In [ ]:
def fill_rate(df):
    total = len(df)
    rates = {}
    for col in df.columns:
        if col == 'document_id':
            continue
        if df[col].dtype in ['int64', 'float64']:
            filled = ((df[col] == 1) | (df[col] == 0)).sum()
        else:
            filled = (df[col].notna() & (df[col] != '') & (df[col] != 'None')).sum()
        rates[col] = f"{100*filled/total:.1f}%" if total > 0 else '0%'
    return rates

html_rates = fill_rate(df_html)
txt_rates = fill_rate(df_txt)

comparison = pd.DataFrame({
    'Column': list(html_rates.keys()),
    'HTML Fill%': list(html_rates.values()),
    'Text Fill%': [txt_rates.get(k, 'N/A') for k in html_rates.keys()]
})
display(comparison)

## 6. Save to CSV

In [ ]:
html_csv = os.path.join(OUTPUT_DIR, 'html_waiver_extraction.csv')
txt_csv  = os.path.join(OUTPUT_DIR, 'text_waiver_extraction.csv')

df_html.to_csv(html_csv, index=False, quoting=csv.QUOTE_ALL)
df_txt.to_csv(txt_csv, index=False, quoting=csv.QUOTE_ALL)

print(f"HTML: {len(df_html)} records saved to {html_csv}")
print(f"Text: {len(df_txt)} records saved to {txt_csv}")

## 7. Preview New Fields

In [ ]:
new_fields = ['document_id', 'title', 'approval_period', 'replacedwaiver', 'waiver_type', 'effective_date']

print("HTML - new fields (first 10):")
display(df_html[new_fields].head(10))

print("\nText - new fields (first 10):")
display(df_txt[new_fields].head(10))